# Reward Router Training: SQL to Training (Second Approach)

This notebook demonstrates the **reward-prediction approach** for VLM routing:

1. Load profiling data from PostgreSQL (real schema: `vlm_samples`, `vlm_responses`, `vlm_evaluations`, `vlm_images`)
2. Compute multi-objective rewards (accuracy, cheap, fast, balanced)
3. Build dataset where each row = (sample, model, mode) with scalar reward target
4. Train a transformer-based router to predict reward for any (query, model, mode) triple
5. Evaluate: compare router vs oracle on selecting best model

**Key Idea**: The router learns to predict how well each model will perform for a given query and mode, then chooses the model with highest predicted reward.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
# Add parent directory to path for imports
import sys
import os

# Get the notebook directory and add parent to path
notebook_dir = os.getcwd()
router_train_dir = os.path.dirname(notebook_dir)
sys.path.insert(0, router_train_dir)

print(f"Notebook dir: {notebook_dir}")
print(f"Router train dir: {router_train_dir}")

In [ ]:
# Standard imports
import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import pearsonr
from tqdm.auto import tqdm
from transformers import AutoTokenizer

In [ ]:


# Project imports
from config import Config, DBConfig, RewardWeights
from db_utils import load_profiles_real_schema, test_connection
from reward_definitions import compute_rewards_real_schema
from models.reward_router import RewardRouterModel
from training.dataset import build_dataloaders, split_by_data_split_column

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Matplotlib settings
plt.style.use('default')
%matplotlib inline

# Set random seeds
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✓ Imports successful")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

## 1. Connect to Database

Connect to PostgreSQL and verify the connection.

In [ ]:
# Load database configuration
db_config = DBConfig.from_env()

# Load database configuration
db_config = DBConfig.from_env()

# Current default: ngrok (remote PACE)
DB_MODE = "ngrok"

# Connections:
if DB_MODE == "ngrok":
    db_config.host = "4.tcp.ngrok.io"
    db_config.port = 16035
else:
    db_config.host = "localhost"
    db_config.port = 5432


print("[DB CONFIG]")
print(f"  Host: {db_config.host}")
print(f"  Port: {db_config.port}")
print(f"  Database: {db_config.name}")
print(f"  User: {db_config.user}")

# Test connection
print("\n[TESTING CONNECTION]")
if test_connection(db_config):
    print("✓ Database connection successful!")
else:
    print("✗ Database connection failed!")
    raise RuntimeError("Could not connect to database")

## 2. Load and Inspect Joined Profiles

Load data from all 4 tables using the real schema:
- `vlm_samples`: sample metadata
- `vlm_responses`: model responses with cost/latency
- `vlm_evaluations`: glider scores (0-5)
- `vlm_images`: image dimensions

In [ ]:
# Load profiles from database
# Set limit=1000 for quick testing, or None for full dataset
LIMIT = None  # Change to e.g. 1000 for quick testing

print("[LOADING PROFILES FROM DB]")
df_profiles = load_profiles_real_schema(
    db_config=db_config,
    limit=LIMIT,
)

print(f"\n[LOADED DATA]")
print(f"  Total rows: {len(df_profiles):,}")
print(f"  Unique samples: {df_profiles['sample_id'].nunique():,}")
print(f"  Unique models: {df_profiles['model_name'].nunique()}")
print(f"  Columns: {list(df_profiles.columns)}")

In [ ]:
# Inspect first few rows
display(df_profiles.head())

In [ ]:
# Dataset statistics
print("[DATASET STATISTICS]")
print("\nModels:")
print(df_profiles['model_name'].value_counts())

print("\nSource datasets:")
print(df_profiles['source_dataset'].value_counts())

print("\nRouter tasks:")
print(df_profiles['router_task'].value_counts())

if 'data_split' in df_profiles.columns:
    print("\nData splits:")
    print(df_profiles['data_split'].value_counts())

In [ ]:
# Check glider_score distribution (our primary accuracy signal)
print("[GLIDER SCORE DISTRIBUTION]")
print(df_profiles['glider_score'].describe())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df_profiles['glider_score'].dropna(), bins=50, edgecolor='black')
plt.xlabel('Glider Score (0-5)')
plt.ylabel('Count')
plt.title('Distribution of Glider Scores')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
df_profiles.groupby('model_name')['glider_score'].mean().sort_values().plot(kind='barh')
plt.xlabel('Mean Glider Score')
plt.title('Average Glider Score by Model')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Check cost and latency distributions
print("[COST AND LATENCY DISTRIBUTIONS]")
print("\nCost (USD):")
print(df_profiles['cost_usd'].describe())
print("\nLatency (ms):")
print(df_profiles['latency_ms'].describe())

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Cost histogram
axes[0, 0].hist(df_profiles['cost_usd'].dropna(), bins=50, edgecolor='black')
axes[0, 0].set_xlabel('Cost (USD)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Cost Distribution')
axes[0, 0].grid(alpha=0.3)

# Latency histogram
axes[0, 1].hist(df_profiles['latency_ms'].dropna(), bins=50, edgecolor='black')
axes[0, 1].set_xlabel('Latency (ms)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Latency Distribution')
axes[0, 1].grid(alpha=0.3)

# Cost by model
df_profiles.groupby('model_name')['cost_usd'].mean().sort_values().plot(kind='barh', ax=axes[1, 0])
axes[1, 0].set_xlabel('Mean Cost (USD)')
axes[1, 0].set_title('Average Cost by Model')
axes[1, 0].grid(alpha=0.3)

# Latency by model
df_profiles.groupby('model_name')['latency_ms'].mean().sort_values().plot(kind='barh', ax=axes[1, 1])
axes[1, 1].set_xlabel('Mean Latency (ms)')
axes[1, 1].set_title('Average Latency by Model')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Compute Rewards & Expand Modes

Compute multi-objective rewards:
- **Accuracy**: $(A^2) \times H$ — maximize quality
- **Cheap**: $A \times H - 0.7 \times (cost\_norm^{1.2})$ — low cost
- **Fast**: $A \times H - 0.7 \times (lat\_norm^{1.2})$ — low latency
- **Balanced**: $(A^2) \times H + 0.3 \times (C^{0.5}) - 0.3 \times (cost^{1.1}) - 0.3 \times (lat^{1.1})$ — multi-objective

Where:
- $A$ = primary accuracy (glider_score / 5)
- $H$ = hallucination cleanliness (1.0, no signal available)
- $C$ = confidence (from confidence_score)
- $cost\_norm$, $lat\_norm$ = normalized cost/latency

In [ ]:
# Initialize reward weights (using defaults)
reward_weights = RewardWeights()

print("[REWARD WEIGHTS]")
print(f"  Accuracy mode: A^{reward_weights.accuracy_exp} * H")
print(f"  Cheap mode: A*H - {reward_weights.cheap_cost_weight}*(cost^{reward_weights.cheap_cost_exp})")
print(f"  Fast mode: A*H - {reward_weights.fast_lat_weight}*(lat^{reward_weights.fast_lat_exp})")
print(f"  Balanced mode: A^{reward_weights.balanced_acc_exp}*H + {reward_weights.balanced_conf_weight}*C^{reward_weights.balanced_conf_exp}")
print(f"                  - {reward_weights.balanced_cost_weight}*cost^{reward_weights.balanced_cost_exp}")
print(f"                  - {reward_weights.balanced_lat_weight}*lat^{reward_weights.balanced_lat_exp}")

In [ ]:
# Compute rewards
print("[COMPUTING REWARDS]")
df_rewards = compute_rewards_real_schema(df_profiles, reward_weights)

print(f"\n[REWARD COLUMNS ADDED]")
reward_cols = ['primary_acc', 'cost_norm', 'lat_norm', 'H', 'C', 
               'reward_accuracy', 'reward_cheap', 'reward_fast', 'reward_balanced']
print(f"  Columns: {reward_cols}")

In [ ]:
# Plot reward distributions
print("[REWARD DISTRIBUTIONS]")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, mode in enumerate(['accuracy', 'cheap', 'fast', 'balanced']):
    col = f'reward_{mode}'
    ax = axes[idx // 2, idx % 2]
    
    ax.hist(df_rewards[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    ax.axvline(df_rewards[col].mean(), color='red', linestyle='--', label=f'Mean: {df_rewards[col].mean():.3f}')
    ax.axvline(df_rewards[col].median(), color='orange', linestyle='--', label=f'Median: {df_rewards[col].median():.3f}')
    ax.set_xlabel('Reward Value')
    ax.set_ylabel('Count')
    ax.set_title(f'{mode.capitalize()} Mode Rewards')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
for mode in ['accuracy', 'cheap', 'fast', 'balanced']:
    col = f'reward_{mode}'
    print(f"\n{mode.upper()}:")
    print(f"  Mean: {df_rewards[col].mean():.4f}")
    print(f"  Std:  {df_rewards[col].std():.4f}")
    print(f"  Min:  {df_rewards[col].min():.4f}")
    print(f"  Max:  {df_rewards[col].max():.4f}")

In [ ]:
# Create model and mode mappings
print("[CREATING MODEL AND MODE MAPPINGS]")

model_names = sorted(df_rewards['model_name'].unique())
model2id = {m: i for i, m in enumerate(model_names)}
id2model = {i: m for m, i in model2id.items()}

mode_names = ['accuracy', 'cheap', 'fast', 'balanced']
mode2id = {m: i for i, m in enumerate(mode_names)}
id2mode = {i: m for m, i in mode2id.items()}

print(f"\nModels ({len(model_names)}):")
for model_name, model_id in model2id.items():
    print(f"  {model_id}: {model_name}")

print(f"\nModes ({len(mode_names)}):")
for mode_name, mode_id in mode2id.items():
    print(f"  {mode_id}: {mode_name}")

In [ ]:
# Add model_id column
df_rewards['model_id'] = df_rewards['model_name'].map(model2id).astype('int64')

# Expand over modes: create one row per (sample, model, mode)
print("[EXPANDING OVER MODES]")
print(f"  Before expansion: {len(df_rewards):,} rows")

rows = []
for mode_name in mode_names:
    mode_id = mode2id[mode_name]
    reward_col = f'reward_{mode_name}'
    
    tmp = df_rewards.copy()
    tmp['mode_name'] = mode_name
    tmp['mode_id'] = mode_id
    tmp['reward'] = tmp[reward_col]
    
    rows.append(tmp)

df_router = pd.concat(rows, ignore_index=True)

print(f"  After expansion: {len(df_router):,} rows ({len(mode_names)} modes)")
print(f"  Unique (sample, model, mode) tuples: {len(df_router.groupby(['sample_id', 'model_id', 'mode_id']))}")

In [ ]:
# Select final training columns
cols_keep = [
    # Sample-level
    'sample_id',
    'prompt_raw',
    'source_config',
    'source_dataset',
    'router_task',
    'data_split',
    'txt_prompt_length_words',
    'txt_prompt_length_chars',
    'img_width',
    'img_height',
    'img_aspect_ratio',
    
    # Model-level
    'model_name',
    'model_id',
    
    # Mode-level
    'mode_name',
    'mode_id',
    
    # Target + debug
    'reward',
    'primary_acc',
    'cost_norm',
    'lat_norm',
    'H',
    'C',
]

# Filter to columns that exist
cols_keep = [c for c in cols_keep if c in df_router.columns]
df_train = df_router[cols_keep].copy()

print("[FINAL DATASET]")
print(f"  Shape: {df_train.shape}")
print(f"  Columns: {list(df_train.columns)}")

display(df_train.head())

In [ ]:
# Save dataset and indices
data_dir = os.path.join(router_train_dir, 'data')
os.makedirs(data_dir, exist_ok=True)

dataset_path = os.path.join(data_dir, 'router_reward_dataset.parquet')
model_index_path = os.path.join(data_dir, 'model_index.json')
mode_index_path = os.path.join(data_dir, 'mode_index.json')

print("[SAVING DATASET]")
df_train.to_parquet(dataset_path, index=False)
print(f"  ✓ Saved dataset: {dataset_path}")

with open(model_index_path, 'w') as f:
    json.dump({'name_to_id': model2id, 'id_to_name': id2model}, f, indent=2)
print(f"  ✓ Saved model index: {model_index_path}")

with open(mode_index_path, 'w') as f:
    json.dump({'name_to_id': mode2id, 'id_to_name': id2mode, 'names': mode_names}, f, indent=2)
print(f"  ✓ Saved mode index: {mode_index_path}")

## 4. Build Dataset & Dataloaders

Split data by `data_split` column (train/val/test) and create PyTorch dataloaders.

In [ ]:
# Split by data_split column
print("[SPLITTING DATASET]")
train_df, val_df, test_df = split_by_data_split_column(df_train)

print(f"\nTrain: {len(train_df):,} rows")
print(f"Val:   {len(val_df):,} rows")
print(f"Test:  {len(test_df):,} rows")

In [ ]:
# Load tokenizer
TEXT_ENCODER = "distilbert-base-uncased"
MAX_SEQ_LENGTH = 256
BATCH_SIZE = 32

print(f"[LOADING TOKENIZER: {TEXT_ENCODER}]")
tokenizer = AutoTokenizer.from_pretrained(TEXT_ENCODER)
print(f"  ✓ Tokenizer loaded")

In [ ]:
# Build dataloaders
print("[BUILDING DATALOADERS]")
train_loader, val_loader, test_loader = build_dataloaders(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    max_seq_length=MAX_SEQ_LENGTH,
    num_workers=0,  # Set to 0 for notebook to avoid multiprocessing issues
    pin_memory=False,
)

print(f"\n[DATALOADERS READY]")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")

In [ ]:
# Inspect a batch
print("[INSPECTING A BATCH]")
batch = next(iter(train_loader))

print(f"  input_ids: {batch['input_ids'].shape}")
print(f"  attention_mask: {batch['attention_mask'].shape}")
print(f"  model_id: {batch['model_id'].shape}")
print(f"  mode_id: {batch['mode_id'].shape}")
print(f"  reward: {batch['reward'].shape}")
print(f"\n  Example decoded text:")
print(f"  {tokenizer.decode(batch['input_ids'][0], skip_special_tokens=True)}")
print(f"\n  Example target reward: {batch['reward'][0].item():.4f}")
print(f"  Example model_id: {batch['model_id'][0].item()} ({id2model[batch['model_id'][0].item()]})")
print(f"  Example mode_id: {batch['mode_id'][0].item()} ({id2mode[batch['mode_id'][0].item()]})")

## 5. Train Reward Router (Second Approach)

Train a transformer-based model that predicts reward for (query, model, mode) triples.

**Architecture**:
1. Text encoder (DistilBERT) → text embedding
2. Model embedding + Mode embedding
3. Concatenate → MLP → scalar reward prediction

**Loss**: MSE between predicted and true reward

In [ ]:
# Training hyperparameters
NUM_EPOCHS = 5
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
GRADIENT_CLIP = 1.0

# Choose the best device (CUDA > MPS > CPU)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("[TRAINING CONFIGURATION]")
print(f"  Num epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Device: {DEVICE}")
print(f"  Text encoder: {TEXT_ENCODER}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")

In [ ]:
# Initialize model
from config import RouterModelConfig

model_config = RouterModelConfig(
    text_encoder_name=TEXT_ENCODER,
    freeze_text_encoder=True,
    model_emb_dim=32,
    mode_emb_dim=16,
    hidden_dim=512,
    num_hidden_layers=2,
    dropout=0.1,
    max_seq_length=MAX_SEQ_LENGTH,
)

num_models = len(model2id)
num_modes = len(mode2id)

print(f"[INITIALIZING MODEL]")
print(f"  Num models: {num_models}")
print(f"  Num modes: {num_modes}")

model = RewardRouterModel(
    config=model_config,
    num_models=num_models,
    num_modes=num_modes,
)

model.to(DEVICE)

print(f"  ✓ Model initialized")
print(f"  Total parameters: {model.count_parameters():,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Setup optimizer and loss
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

criterion = nn.MSELoss()

print("[OPTIMIZER & LOSS]")
print(f"  Optimizer: AdamW")
print(f"  Loss: MSE")

In [ ]:
# Training loop
history = {
    'train_loss': [],
    'val_loss': [],
    'val_pearson': [],
}

print("[TRAINING]\n")

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    
    with tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]") as pbar:
        for batch in pbar:
            # Move to device
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            model_id = batch['model_id'].to(DEVICE)
            mode_id = batch['mode_id'].to(DEVICE)
            reward = batch['reward'].to(DEVICE)
            
            # Forward
            optimizer.zero_grad()
            pred_reward = model(input_ids, attention_mask, model_id, mode_id)
            
            # Loss
            loss = criterion(pred_reward, reward)
            
            # Backward
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    train_loss /= len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        with tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]") as pbar:
            for batch in pbar:
                # Move to device
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                model_id = batch['model_id'].to(DEVICE)
                mode_id = batch['mode_id'].to(DEVICE)
                reward = batch['reward'].to(DEVICE)
                
                # Forward
                pred_reward = model(input_ids, attention_mask, model_id, mode_id)
                
                # Loss
                loss = criterion(pred_reward, reward)
                val_loss += loss.item()
                
                # Collect predictions
                all_preds.extend(pred_reward.cpu().numpy())
                all_targets.extend(reward.cpu().numpy())
                
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    val_loss /= len(val_loader)
    
    # Compute Pearson correlation
    val_pearson, _ = pearsonr(all_preds, all_targets)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_pearson'].append(val_pearson)
    
    # Print epoch summary
    print(f"\n[EPOCH {epoch+1}] train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_pearson={val_pearson:.4f}")
    
    # Show sample predictions
    sample_indices = np.random.choice(len(all_preds), size=min(3, len(all_preds)), replace=False)
    print("  Sample predictions (pred vs true):")
    for idx in sample_indices:
        print(f"    {all_preds[idx]:.4f} vs {all_targets[idx]:.4f}")
    print()

print("\n[TRAINING COMPLETE]")

In [ ]:
# Save trained model
checkpoints_dir = os.path.join(router_train_dir, 'models', 'checkpoints')
os.makedirs(checkpoints_dir, exist_ok=True)

model_save_path = os.path.join(checkpoints_dir, 'best_reward_router.pt')
model.save(model_save_path)
print(f"[MODEL SAVED] {model_save_path}")

## 6. Plots & Diagnostics

Visualize training progress and evaluate router performance.

In [ ]:
# Plot training curves
print("[TRAINING CURVES]")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
axes[0].plot(history['train_loss'], marker='o', label='Train Loss')
axes[0].plot(history['val_loss'], marker='s', label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Pearson correlation
axes[1].plot(history['val_pearson'], marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Pearson Correlation')
axes[1].set_title('Validation Pearson Correlation (pred vs true reward)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Val Loss: {history['val_loss'][-1]:.4f}")
print(f"Final Val Pearson: {history['val_pearson'][-1]:.4f}")

In [ ]:
# Get predictions on validation set
print("[VALIDATION PREDICTIONS]")

model.eval()
val_preds = []
val_targets = []
val_sample_ids = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Predicting on val set"):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        model_id = batch['model_id'].to(DEVICE)
        mode_id = batch['mode_id'].to(DEVICE)
        reward = batch['reward'].to(DEVICE)
        
        pred_reward = model(input_ids, attention_mask, model_id, mode_id)
        
        val_preds.extend(pred_reward.cpu().numpy())
        val_targets.extend(reward.cpu().numpy())
        val_sample_ids.extend(batch['sample_ids'])

val_preds = np.array(val_preds)
val_targets = np.array(val_targets)

print(f"  Collected {len(val_preds)} predictions")

In [ ]:
# Scatter plot: predicted vs true reward
print("[PREDICTED VS TRUE REWARD]")

# Random subsample for clarity
n_plot = min(5000, len(val_preds))
indices = np.random.choice(len(val_preds), size=n_plot, replace=False)

plt.figure(figsize=(8, 8))
plt.scatter(val_targets[indices], val_preds[indices], alpha=0.3, s=10)
plt.plot([val_targets.min(), val_targets.max()], 
         [val_targets.min(), val_targets.max()], 
         'r--', label='Perfect prediction')
plt.xlabel('True Reward')
plt.ylabel('Predicted Reward')
plt.title(f'Predicted vs True Reward (Val Set, n={n_plot})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

corr, _ = pearsonr(val_preds, val_targets)
print(f"\nPearson correlation: {corr:.4f}")

In [ ]:
# Add predictions to val_df for oracle comparison
print("[ADDING PREDICTIONS TO VAL DF]")

val_df_with_preds = val_df.copy()
val_df_with_preds['pred_reward'] = val_preds

print(f"  Val df shape: {val_df_with_preds.shape}")
display(val_df_with_preds[['sample_id', 'model_name', 'mode_name', 'reward', 'pred_reward']].head(10))

In [ ]:
# Oracle vs Router comparison on a small subset
print("[ORACLE VS ROUTER COMPARISON]")

# For each (sample_id, mode_id), find:
#   - Oracle: model with max TRUE reward
#   - Router: model with max PREDICTED reward

results = []
unique_samples = val_df_with_preds['sample_id'].unique()[:100]  # Sample 100 for quick demo

for sample_id in tqdm(unique_samples, desc="Comparing oracle vs router"):
    sample_data = val_df_with_preds[val_df_with_preds['sample_id'] == sample_id]
    
    for mode_id in range(num_modes):
        mode_name = id2mode[mode_id]
        mode_data = sample_data[sample_data['mode_id'] == mode_id]
        
        if len(mode_data) == 0:
            continue
        
        # Oracle: max true reward
        oracle_idx = mode_data['reward'].idxmax()
        oracle_row = mode_data.loc[oracle_idx]
        
        # Router: max predicted reward
        router_idx = mode_data['pred_reward'].idxmax()
        router_row = mode_data.loc[router_idx]
        
        match = oracle_row['model_name'] == router_row['model_name']
        
        results.append({
            'sample_id': sample_id,
            'mode_name': mode_name,
            'oracle_model': oracle_row['model_name'],
            'oracle_reward': oracle_row['reward'],
            'router_model': router_row['model_name'],
            'router_pred_reward': router_row['pred_reward'],
            'router_true_reward': router_row['reward'],  # What the router actually got
            'match': match,
        })

results_df = pd.DataFrame(results)

print(f"\n[RESULTS ON {len(unique_samples)} SAMPLES]")
print(f"  Total decisions: {len(results_df)}")
print(f"  Router matches oracle: {results_df['match'].sum()} / {len(results_df)} ({100*results_df['match'].mean():.1f}%)")

# Show by mode
print("\n  By mode:")
for mode_name in mode_names:
    mode_results = results_df[results_df['mode_name'] == mode_name]
    if len(mode_results) > 0:
        match_rate = mode_results['match'].mean()
        print(f"    {mode_name:12s}: {100*match_rate:.1f}% match")

# Show a few examples
print("\n  Example comparisons:")
display(results_df.head(10))

In [ ]:
# Plot routing accuracy by mode
print("[ROUTING ACCURACY BY MODE]")

match_rates = results_df.groupby('mode_name')['match'].mean()

plt.figure(figsize=(10, 5))
match_rates.plot(kind='bar', color=['green', 'blue', 'orange', 'purple'])
plt.ylabel('Routing Accuracy (% match with oracle)')
plt.title('Router vs Oracle Agreement by Mode')
plt.xlabel('Reward Mode')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Higher is better (router agrees with oracle more often)")
print("  - Perfect score (1.0) = router always picks same model as oracle")
print(f"  - Current overall accuracy: {100*results_df['match'].mean():.1f}%")

In [ ]:
# Analyze router's reward recovery
print("[REWARD RECOVERY ANALYSIS]")

# How much reward does the router get vs oracle?
reward_comparison = results_df.groupby('mode_name').agg({
    'oracle_reward': 'mean',
    'router_true_reward': 'mean',
}).reset_index()

reward_comparison['reward_ratio'] = reward_comparison['router_true_reward'] / reward_comparison['oracle_reward']

print("\nAverage rewards:")
display(reward_comparison)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(reward_comparison))
width = 0.35

ax.bar(x - width/2, reward_comparison['oracle_reward'], width, label='Oracle', color='green', alpha=0.7)
ax.bar(x + width/2, reward_comparison['router_true_reward'], width, label='Router', color='blue', alpha=0.7)

ax.set_xlabel('Mode')
ax.set_ylabel('Average Reward')
ax.set_title('Oracle vs Router Average Reward by Mode')
ax.set_xticks(x)
ax.set_xticklabels(reward_comparison['mode_name'])
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nReward recovery (router / oracle):")
for _, row in reward_comparison.iterrows():
    print(f"  {row['mode_name']:12s}: {100*row['reward_ratio']:.1f}%")

## Summary

This notebook demonstrated the **reward-prediction approach** for VLM routing:

1. ✅ Connected to PostgreSQL and loaded real profiling data
2. ✅ Computed multi-objective rewards (accuracy, cheap, fast, balanced)
3. ✅ Built dataset with (sample, model, mode) → reward mapping
4. ✅ Trained transformer router to predict rewards
5. ✅ Evaluated router vs oracle

**Key Results**:
- Router learns to predict rewards with **Pearson correlation ≈ 0.XX** (check val_pearson above)
- Router matches oracle's model choice **XX%** of the time on average
- Router recovers **XX%** of oracle's reward on average

**Next Steps**:
- Tune reward weights in `RewardWeights` config
- Train for more epochs
- Try different text encoders (BERT, RoBERTa)
- Add image features to the router input
- Evaluate on full test set
- Compare against baselines (always biggest, always cheapest, random)